# P7: Naive Bayes Spam Classifier

**FIT1061 Introduction to Artificial Intelligence — Week 7**

Last week you computed P(spam | word) for individual words. This week you build a **classifier** — a machine that reads an entire email and decides: spam or not spam? In 2016, ProPublica revealed that COMPAS — a risk assessment algorithm used in US courts — was biased against Black defendants. The same mathematics powers both spam filters and bail decisions.

**This notebook covers P7.2 (implementation) and P7.3 (evaluation).** Make sure you have completed P7.1 (hand computation) before starting.

> **🗣 Milestone task.** P7 requires a tutor discussion for sign-off. After completing all parts (including the hand computation and reflection), book a discussion with your tutor.

> **P7.1 is personalised by your Monash student ID** — see the 'Your personalised instance' section below.


---

## Setup

In [1]:
from p7_helpers import (TRAIN_DATA, TEST_DATA, HAND_TRACE_DATA,
                        HAND_TRACE_TEST, tokenise, build_vocabulary,
                        print_dataset_summary, print_confusion_matrix,
                        plot_log_score_bars)
print("Imports OK.")

Imports OK.


---

## Warm-up: the trace from the whiteboard

In the lecture we classified the message **"free meeting"** by hand (and watched a single zero
wipe out a strong spam signal). Work through it yourself to warm up, then classify your own test
message in Part 0.

Stuck? This is the exact example from the lecture: revisit the slides or the workshop.

**Training set** (the same six emails as Part 0):

| # | label | words |
|---|---|---|
| 1 | spam | free, prize, winner |
| 2 | spam | free, offer, click |
| 3 | spam | winner, call, now |
| 4 | ham | meeting, agenda, tuesday |
| 5 | ham | project, update, meeting |
| 6 | ham | free, lunch, friday |

**Test message: "free meeting."** By hand:

1. **Priors:** P(spam) = ?, P(ham) = ?
2. **Likelihoods** (fraction of messages in each class that contain the word):
   P(free | spam) = ?, P(free | ham) = ?, P(meeting | spam) = ?, P(meeting | ham) = ?
3. **Scores:** score(spam) = P(spam) x P(free|spam) x P(meeting|spam) = ?; score(ham) = ?
4. **Prediction:** which class wins? Is it the one you'd expect from the word "free"?

*(Same method as Part 0 below; there you classify YOUR own personalised test message.)*


---

## Your personalised instance (P7.1)

The hand-trace test email is **personalised** by your Monash student ID. Each student gets a different test email engineered to surface the zero-frequency property of Naive Bayes (half the cohort sees a spam-side zero, half sees a ham-side zero). Your trace will differ from a peer's.

**Set your Monash student ID below**, then run the next two cells. Do not edit `task_instances.py`.


In [2]:
# Replace the placeholder with your Monash student ID (8 digits).
STUDENT_ID = "37524720"

assert STUDENT_ID.isdigit() and len(STUDENT_ID) == 8, (
    "Set STUDENT_ID to your 8-digit Monash student ID (digits only, in quotes)."
)


In [3]:
# Integrity check + instance generation.
# DO NOT EDIT task_instances.py — your tutor regenerates from your ID
# and compares fingerprints. Modifications will be caught.

from task_instances import generate_instance
from task_instances_lib import fingerprint

instance = generate_instance(STUDENT_ID)
email_inst = instance["email"]

print(f"Student ID:                {STUDENT_ID}")
print(f"task_instances.py SHA-256: {fingerprint('task_instances.py')}")
print()
print(email_inst.describe())


Student ID:                37524720
task_instances.py SHA-256: 1f3f1cb49bcd49ccd35dab001ea651ca50a71537bcddf23c362c7d7afd75ae6f

Personalised test email for student 37524720
  email_text   = "free now"
  n_words      = 2
  wipeout_side = ham  (the class with a zero-frequency word)


---

## Part 0: The hand-trace dataset (P7.1)

Before coding, you should trace one Naive Bayes classification **by hand** on paper (P7.1). This is the most important part of the task: if you understand the hand trace, the code will make sense. If you skip it, the code will feel like magic.

The training set (`HAND_TRACE_DATA`) is the same for everyone: six emails, three spam and three ham. **Your test email is personalised** by your Monash student ID. Different students get different test emails, and each is engineered to expose a key property of Naive Bayes (see Part 1).

### What is a hand trace?

Naive Bayes classification is a **tree of multiplications**. You start at the top with your prior belief ("how likely is spam in general?"), then each word in the email updates that belief multiplicatively. Your hand trace shows the tree.

Run the cell below to see the training data + YOUR personalised test email.


In [4]:
print("Hand-trace training data (HAND_TRACE_DATA — same for everyone):")
print()
for i, (label, text) in enumerate(HAND_TRACE_DATA, 1):
    print(f"  {i}. [{label:4s}] {text}")

print(f"\n{'='*50}")
print(f"YOUR personalised test email: \"{email_inst.email_text}\"")
print(f"Words to multiply: {tokenise(email_inst.email_text)}")
print(f"{'='*50}")

print("\nOn paper, work through the tree:")
print()
print("  TOP:    Compute priors from training data")
print("    • P(spam) = ? / 6")
print("    • P(ham)  = ? / 6")
print()
print("  MIDDLE: For each word in YOUR test email, compute P(word|spam) and P(word|ham)")
print("    • Count how many spam emails contain that word, divide by 3")
print("    • Count how many ham  emails contain that word, divide by 3")
print()
print("  BOTTOM: Multiply prior × each likelihood for each class → two scores")
print("    • score_spam = P(spam) × P(w1|spam) × P(w2|spam) × ...")
print("    • score_ham  = P(ham)  × P(w1|ham)  × P(w2|ham)  × ...")
print()
print("  PREDICTION: class with the higher score wins.")


Hand-trace training data (HAND_TRACE_DATA — same for everyone):

  1. [spam] free prize winner
  2. [spam] free offer click
  3. [spam] winner call now
  4. [ham ] meeting agenda tuesday
  5. [ham ] project update meeting
  6. [ham ] free lunch friday

YOUR personalised test email: "free now"
Words to multiply: {'free', 'now'}

On paper, work through the tree:

  TOP:    Compute priors from training data
    • P(spam) = ? / 6
    • P(ham)  = ? / 6

  MIDDLE: For each word in YOUR test email, compute P(word|spam) and P(word|ham)
    • Count how many spam emails contain that word, divide by 3
    • Count how many ham  emails contain that word, divide by 3

  BOTTOM: Multiply prior × each likelihood for each class → two scores
    • score_spam = P(spam) × P(w1|spam) × P(w2|spam) × ...
    • score_ham  = P(ham)  × P(w1|ham)  × P(w2|ham)  × ...

  PREDICTION: class with the higher score wins.


---

### Articulation (P7.1: graded with the photo)

Show your full hand trace on paper for YOUR test email. Then fill in the answer cell below.

Compute by hand (using the HAND_TRACE_DATA training set above):

1. **Priors:** $P(\text{spam}) = ?/6$, $P(\text{ham}) = ?/6$
2. **Per-word likelihoods** for each word in your test email, separately for `spam` and `ham` (fractions like `2/3`, `0/3`, etc.)
3. **Scores:** multiply prior × all likelihoods for each class.
4. **Prediction.**
5. **Did one class score collapse to 0?** Which class, and which word killed it?


In [ ]:
# Articulation answers — type your hand-computed values here.
# Reference YOUR test email words. Show fractions or 4-decimal floats.

answers = {
    "prior_spam": None,             # e.g. 0.5
    "prior_ham":  None,             # e.g. 0.5
    "score_spam": None,             # e.g. 0.0 or 0.037037
    "score_ham":  None,             # e.g. 0.0 or 0.111111
    "prediction": None,             # "spam" or "ham"
    "zero_frequency_killed": None,  # True or False
    "killed_side": None,            # "spam", "ham", or "" if no kill
}

# Brief explanation (2-3 sentences) — which word(s) drove your result?
# Reference the zero-frequency phenomenon if it applies:
explanation = (
    "TYPE YOUR EXPLANATION HERE."
)

for k, v in answers.items():
    assert v is not None, f"answers['{k}'] is still None"
assert explanation.strip() != "TYPE YOUR EXPLANATION HERE.", "Write your explanation."
print("Filled in. Photograph hand trace + this cell for P7.1.")


> **Did one branch die?** If one of your two scores came out to 0, that's correct. Look at your tree: a word in your test email had zero count in one of the classes, so its likelihood was 0. Multiplying by zero kills the entire branch, no matter how strongly the other words pointed elsewhere. This is the **zero-frequency problem**, and you'll see it again in Part 5.
>
> Some students see the **spam** branch die (a ham-only word like *meeting* or *agenda*). Other students see the **ham** branch die (a spam-only word like *prize* or *winner*). Compare with a peer.


---

## Part 1: Explore the main dataset

Now let's look at the larger dataset you'll use for the classifier.

In [ ]:
print_dataset_summary(TRAIN_DATA, "Training set")
print_dataset_summary(TEST_DATA, "Test set")

print("\nSample spam:")
for label, text in TRAIN_DATA[:3]:
    print(f"  {text}")

print("\nSample ham:")
for label, text in TRAIN_DATA[12:15]:
    print(f"  {text}")

vocab = build_vocabulary(TRAIN_DATA)
print(f"\nVocabulary: {len(vocab)} unique words")

---

## Part 2: Implement Naive Bayes (P7.2)

You'll implement three functions. Each one does exactly what you did by hand in P7.1, just on a bigger dataset.

### 2a. Compute priors

The **prior** is the probability of each class before looking at any words. This is the same calculation you did in P6.

In [ ]:
def compute_priors(data):
    """Compute P(spam) and P(ham) from training data.

    Args:
        data: List of (label, text) tuples.

    Returns:
        A dict {"spam": float, "ham": float}.
    """
    total = len(data)
    spam_count = sum(1 for label, _ in data if label == "spam")
    ham_count = total - spam_count
    return {"spam": spam_count / total, "ham": ham_count / total}


# Test it
priors = compute_priors(TRAIN_DATA)
print(f"P(spam) = {priors['spam']:.4f}")
print(f"P(ham)  = {priors['ham']:.4f}")
print(f"Sum:      {priors['spam'] + priors['ham']:.4f}")  # Should be 1.0

### Stepping stone: one word at a time

Before computing likelihoods for *every* word, look at three words by hand using the training data.

**Fill in this table by counting** (you can scan `TRAIN_DATA` mentally or use `print(TRAIN_DATA)` to inspect):

| word     | # spam emails containing it | # ham emails containing it | P(word \| spam) = count / 12 | P(word \| ham) = count / 20 |
|----------|----------------------------:|---------------------------:|----------------------------:|---------------------------:|
| `free`   |  ?                          |  ?                         |  ?                          |  ?                         |
| `meeting`|  ?                          |  ?                         |  ?                          |  ?                         |
| `winner` |  ?                          |  ?                         |  ?                          |  ?                         |

(Reminder: the training set has 12 spam emails and 20 ham emails; see the dataset summary printed above.)

You should see that `free` and `winner` are much higher for spam, and `meeting` is much higher for ham: the likelihoods that drive the classifier.

**Then think about how you would generalise this to all words at once** before writing the code in the next cell.

> **Self-check** (after you have filled in the table above).
>
> | word     | spam count | ham count | P(w \| spam)  | P(w \| ham)  |
> |----------|-----------:|----------:|-------------:|------------:|
> | `free`   | (compute this) | (compute this) | (compute this) | (compute this) |
> | `meeting`| (compute this) | (compute this) | (compute this) | (compute this) |
> | `winner` | (compute this) | (compute this) | (compute this) | (compute this) |
>
> Once you have filled it in, look closely at your likelihoods: at least one of these words never appeared in one of the two classes during training, so its likelihood there comes out to **zero**. Hold onto that observation; it is the lesson of Part 5.
>
> The next cell asks you to compute the full likelihood table for every word, over a thousand `(word, class)` cells. Think about the right shape of data structure before you start writing.

In [ ]:
def compute_likelihoods(data, vocabulary):
    """Compute P(word | class) for every word and class.

    Args:
        data: List of (label, text) tuples.
        vocabulary: List of all words.

    Returns:
        A nested dict: {"spam": {"word1": float, ...}, "ham": {...}}
    """
    # --- YOUR CODE HERE ---
    # For each class (spam, ham):
    #   1. Get all messages with that label
    #   2. For each word in the vocabulary:
    #      Count how many messages in that class contain the word
    #      Divide by total messages in that class


    # --- END YOUR CODE ---


# Test it
likelihoods = compute_likelihoods(TRAIN_DATA, vocab)

# Show some informative words
print(f"{'Word':>15s}  {'P(w|spam)':>10s}  {'P(w|ham)':>10s}")
print("-" * 40)
for word in ["free", "prize", "meeting", "project", "call", "lunch"]:
    if word in likelihoods["spam"]:
        ps = likelihoods["spam"][word]
        ph = likelihoods["ham"][word]
        print(f"{word:>15s}  {ps:>10.4f}  {ph:>10.4f}")

**Check:** "free" should be much higher for spam than ham. "meeting" and "project" should be higher for ham. If your numbers look wrong, check your counting logic.

### 2c. Predict

The **predict** function classifies one email by computing the Naive Bayes score for each class:

$$\text{score}_{\text{spam}} = P(\text{spam}) \times \prod_{\text{word} \in \text{email}} P(\text{word} | \text{spam})$$

$$\text{score}_{\text{ham}} = P(\text{ham}) \times \prod_{\text{word} \in \text{email}} P(\text{word} | \text{ham})$$

Predict the class with the higher score. Only use words that are in the vocabulary (skip unknown words).

In [ ]:
def predict(text, priors, likelihoods, vocabulary):
    """Classify one email using Naive Bayes.

    Args:
        text: The email text (a string).
        priors: Dict from compute_priors.
        likelihoods: Dict from compute_likelihoods.
        vocabulary: List of known words.

    Returns:
        "spam" or "ham"
    """
    words = tokenise(text)

    # --- YOUR CODE HERE ---
    # For each class, compute the Naive Bayes score:
    #
    #     score(class) = P(class)  *  product of P(word | class)
    #                                  for each word in the email
    #                                  that is in the vocabulary.
    #
    # Skip words that are not in the vocabulary.
    # Return the class with the higher score.


    # --- END YOUR CODE ---


# Test on obvious examples
print("Test predictions:")
test_messages = [
    "free prize winner call now",
    "meeting tomorrow at 3pm",
    "free lunch on friday",
]
for msg in test_messages:
    pred = predict(msg, priors, likelihoods, vocab)
    print(f"  [{pred:4s}] {msg}")

**Check:** "free prize winner call now" should be spam. "meeting tomorrow at 3pm" should be ham. "free lunch on friday" could go either way; it's an interesting edge case.

---

## Part 3: Run on the test set

Now classify all test emails and compute accuracy.

In [ ]:
correct = 0
predictions = []

print(f"{"Actual":>6s}  {"Predicted":>9s}  {"":>3s}  Message")
print("-" * 70)

for actual, text in TEST_DATA:
    pred = predict(text, priors, likelihoods, vocab)
    predictions.append(pred)
    match = "OK" if pred == actual else "WRONG"
    if pred == actual:
        correct += 1
    print(f"{actual:>6s}  {pred:>9s}  {match:>5s}  {text}")

accuracy = correct / len(TEST_DATA)
print(f"
Accuracy: {correct}/{len(TEST_DATA)} = {accuracy:.0%}")


---

## Part 4: Evaluation: Confusion matrix, precision, recall (P7.3)

Accuracy tells you the overall success rate. But for spam filtering (and especially for criminal justice) you need to know **what kind** of errors the classifier makes.

- **False positive (FP):** A ham email flagged as spam. Your friend's email goes to junk.
- **False negative (FN):** A spam email that got through. Junk reaches your inbox.

### 4a. Build the confusion matrix

In [ ]:
# --- YOUR CODE HERE ---

# Count TP, FP, FN, TN by comparing actual labels with predictions.
# TP: actual=spam, predicted=spam
# FP: actual=ham,  predicted=spam
# FN: actual=spam, predicted=ham
# TN: actual=ham,  predicted=ham

tp = 0
fp = 0
fn = 0
tn = 0


# --- END YOUR CODE ---

print_confusion_matrix(tp, fp, fn, tn)

### 4b. Compute precision and recall

In [ ]:
# --- YOUR CODE HERE ---

precision = None  # Replace
recall = None     # Replace

# --- END YOUR CODE ---

print(f"Precision: {precision:.2%}")
print(f"  → Of the emails flagged as spam, {precision:.0%} really were spam.")
print(f"Recall: {recall:.2%}")
print(f"  → Of all actual spam, we caught {recall:.0%}.")

### 4c. Inspect misclassified messages

Look at each message the classifier got wrong. For each one, write a 1–2 sentence hypothesis: **why** did the classifier make this mistake? What words might have misled it?

In [ ]:
# zip pairs up each test email with its prediction so we can compare them
print("Misclassified messages:\n")
misclassified = 0

for (actual, text), pred in zip(TEST_DATA, predictions):
    if pred != actual:
        misclassified += 1
        error_type = "FALSE POSITIVE" if pred == "spam" else "FALSE NEGATIVE"
        print(f"{misclassified}. [{error_type}]")
        print(f"   Actual: {actual}, Predicted: {pred}")
        print(f"   Text: \"{text}\"")
        print(f"   Words: {tokenise(text)}")
        print()

if misclassified == 0:
    print("No errors on this test set! The classifier got all 8 correct.")
    print("In your reflection, discuss: does 100% on 8 messages mean")
    print("the classifier is perfect? What could go wrong on new data?")

**For each misclassified message** (or if none, for a message that was close), write a hypothesis:

_Your hypotheses here._

---

### P7.4: Reflection: "What safeguards would you demand for an AI bail system?" (150–250 words)

Write a short reflective response (150–250 words):

> *"What safeguards would you demand for an AI bail system?"*

COMPAS is a real algorithm used in real courtrooms to predict recidivism (re-offending). ProPublica's analysis found it was twice as likely to falsely flag Black defendants as high-risk. Think about your confusion matrix. What happens when the "false positive" means someone stays in jail?

This week ran a Kialo debate on the motion *"Risk scores have no place in bail decisions."* A strong reflection is **expected** to engage a peer's argument from that debate, not only your own view; this is a quality dimension of the reflection rubric, **not a submission requirement**. You can submit on time either way; a reflection that ignores the debate may simply be returned for revision.

**Write your response in the cell below — it stays in this notebook.** You do not upload it as a separate file.

---

**Your response:**

*(Write here)*

---

## Part 4d: Why did the classifier decide that? (graded)

The numbers from Part 4 tell you *how often* the classifier is right. They do not tell you *why* it was right or wrong on any specific email. For your milestone discussion, you should be able to explain a single prediction word by word.

**Insight.** The Naive Bayes score is a product:

$$\text{score}(c) = P(c) \cdot \prod_{w \in \text{email}} P(w \mid c)$$

Multiplications are hard to compare visually. Take the log of both sides:

$$\log \text{score}(c) = \log P(c) + \sum_{w \in \text{email}} \log P(w \mid c)$$

Now each word *adds* a contribution to the log-score. A bar chart of those contributions, with one bar per word per class, shows you exactly which words moved the prediction.

**Your task.** Implement `plot_prediction_trace(text, priors, likelihoods, vocabulary)`. Use the natural log. Pass the per-word contributions to the provided render helper `plot_log_score_bars`; it handles the styling. You do the computation.

**Skip the zero-probability problem for now.** If a word has likelihood exactly 0, skip it in the trace (we will return to that case in Part 5). In real code you would use smoothing; Part 5 motivates why.

In [ ]:
import math

def plot_prediction_trace(text, priors, likelihoods, vocabulary,
                          actual_label=None, title=None):
    """Render a per-word log-contribution bar chart for one email.

    Args:
        text: email string.
        priors: dict from compute_priors.
        likelihoods: dict from compute_likelihoods.
        vocabulary: list of vocab words.
        actual_label: optional "spam"/"ham" — annotates correctness on the plot.
        title: optional plot title.
    """
    words = tokenise(text)

    # --- YOUR CODE HERE ---
    # Build four lists of contributions for the helper:
    #
    #   tracked_words      -- words from the email that are in vocabulary
    #                         AND have non-zero likelihood in BOTH classes
    #   log_spam_contribs  -- log P(word | spam) for each tracked word
    #   log_ham_contribs   -- log P(word | ham)  for each tracked word
    #
    # Also compute the two scalar log priors:
    #   log_prior_spam = log P(spam)
    #   log_prior_ham  = log P(ham)
    #
    # Then call predict(text, priors, likelihoods, vocabulary) to get the
    # model prediction. Finally pass everything to plot_log_score_bars.
    # See the docstring of plot_log_score_bars for the expected arguments.

    raise NotImplementedError("Replace this raise statement with your code")
    # --- END YOUR CODE ---


# Try it on a confident correct prediction (a ham email from the test set):
plot_prediction_trace(
    "see you at the meeting tomorrow morning",
    priors, likelihoods, vocab,
    actual_label="ham",
    title="Confident-correct example",
)

# Now try it on a misclassified email if you have one from Part 4c;
# otherwise pick a borderline case from the test set yourself:
plot_prediction_trace(
    "you won a free holiday reply to claim your prize now",
    priors, likelihoods, vocab,
    actual_label="spam",
    title="Choose one of your interesting Part 4c emails",
)

**P7.4d: Interpret your two plots** (200 words total).

For *each* of the two emails you traced:

1. Which **one or two words** moved the decision the most? Why those words?
2. Did the **prior** matter more or less than the words?
3. If the prediction was wrong, which words would you need to *down-weight* (or which extra training data would you need) to flip it?

_Your answer here._

> **Why this matters for the milestone.** Plenty of P7 students can copy the Bayes formula. Few can point at the bar chart and say "this `free` bar is what made it think spam, and the prior didn't have much to say." That is the difference your tutor will probe.

---

## Part 5: The zero-frequency problem

Try classifying this tricky message:

In [ ]:
tricky = "free meeting about the project"
pred = predict(tricky, priors, likelihoods, vocab)
print(f'Classifying: "{tricky}"')
print(f"Prediction: {pred}")

# Look at the per-word likelihoods.
print(f"\nPer-word likelihoods (training):")
words = tokenise(tricky)
print(f"  {'word':>10s}  {'P(w|spam)':>10s}  {'P(w|ham)':>10s}  flag")
print(f"  {'-'*10}  {'-'*10}  {'-'*10}  ----")
for w in sorted(words):
    if w in likelihoods["spam"]:
        ps = likelihoods["spam"][w]
        ph = likelihoods["ham"][w]
        flag = "ZERO!" if (ps == 0 or ph == 0) else ""
        print(f"  {w:>10s}  {ps:>10.4f}  {ph:>10.4f}  {flag}")
    else:
        print(f"  {w:>10s}  (not in vocab)")

**Question:** Did the model still produce a prediction? Look at the per-word table above: one or more words have likelihood zero in one class. Which word(s) caused trouble, and what would happen to the *score* (product of likelihoods) if you did not skip zero-probability words?

This is the **zero-frequency problem**: if a word never appears in a class during training, its likelihood is 0, which zeroes out the entire score, no matter how much other evidence there is. The trace in Part 4d sidestepped this by ignoring those words, but in a real classifier you cannot silently drop evidence.

The fix is **Laplace smoothing**: adding a small count to every word-class combination so no probability is ever exactly zero. This is a Credit-level extension (see C2 Classification & Fairness).

_Your answer here._

---

## Part 6: Record your results

Fill in the table below (double-click to edit).

| Metric | Value |
|--------|-------|
| Accuracy | ___% |
| True Positives | ___ |
| False Positives | ___ |
| False Negatives | ___ |
| True Negatives | ___ |
| Precision | ___% |
| Recall | ___% |

**Answer:** Which is more important for a spam filter, precision or recall? Why?

_Your answer here._

---

## Submission checklist

- [ ] P7.1: Hand computation of Naive Bayes (photographed) — this is your single appended file
- [ ] P7.2: `compute_priors`, `compute_likelihoods`, and `predict` all working (Parts 2–3)
- [ ] P7.3: Confusion matrix, precision, recall computed (Part 4)
- [ ] Misclassified messages inspected with hypotheses (Part 4c)
- [ ] P7.4d: Prediction trace on two emails + 200-word interpretation (Part 4d)
- [ ] Zero-frequency problem observed and explained (Part 5)
- [ ] Results table filled in (Part 6)
- [ ] P7.4: Reflection on "What safeguards would you demand?" (150–250 words, written in this notebook)
- [ ] Milestone discussion booked with tutor

**Uploading to OnTrack.** Submit **two things**: (1) this completed notebook, and (2) a **single** photo or PDF of your hand computation (P7.1). Your P7.4 reflection stays *inside this notebook* — OnTrack allows only one extra file besides the notebook, and that file is your hand computation.